In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, schema=None):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    # Structured outputs replace the old "prefill + stop sequence" trick:
    # the API guarantees the response text is JSON matching this schema.
    if schema:
        params["output_config"] = {"format": {"type": "json_schema", "schema": schema}}

    message = client.messages.create(**params)
    # The model may think first, so pick the text blocks instead of content[0]
    return "".join(block.text for block in message.content if block.type == "text")

In [3]:
import json

# Shape we want back. The top level must be an object, so the array of
# tasks lives under a "tasks" key.
dataset_schema = {
    "type": "object",
    "properties": {
        "tasks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"task": {"type": "string"}},
                "required": ["task"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["tasks"],
    "additionalProperties": False,
}


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, schema=dataset_schema)
    return json.loads(text)["tasks"]

In [4]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    prompt = f"""
Please solve this following task:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    #TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [7]:
def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [8]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a single function that counts all objects in an S3 bucket, using a boto3 paginator so it works correctly for buckets with more than 1,000 objects (the per-request limit of `list_objects_v2`):\n\n```python\nimport boto3\n\n\ndef count_s3_objects(bucket_name: str) -> int:\n    \"\"\"\n    Return the total number of objects stored in the given S3 bucket.\n\n    Uses a paginator so that all pages of results are traversed,\n    regardless of how many objects the bucket contains.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n\n    Returns:\n        The total number of objects in the bucket.\n    \"\"\"\n    s3 = boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n\n    total = 0\n    for page in paginator.paginate(Bucket=bucket_name):\n        # 'Contents' is absent from a page when it has no objects\n        total += len(page.get(\"Contents\", []))\n\n    return total\n\n\nif __name__ == \"__main__\":\n    print(count_s3_